# Mood Model — FER2013 Training (Colab)

Train **MobileNetV3-Small** on FER2013, then download **`best_model.pt`** to:

`smartshop/mood_model/artifacts/models/best_model.pt`

## Steps
1. **Runtime → Change runtime type → GPU** (recommended)
2. Run all cells in order
3. Upload **`kaggle.json`** when prompted ([Kaggle API token](https://www.kaggle.com/settings))
4. Download `best_model.pt` from the last cell

In [ ]:
# 1) Install dependencies
!pip -q install torch torchvision scikit-learn matplotlib PyYAML opencv-python-headless

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2) (Optional) Mount Google Drive — keeps data between sessions
USE_DRIVE = False  # set True to mount Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/mood_model_colab"
else:
    BASE_DIR = "/content/mood_model_colab"

from pathlib import Path
BASE_DIR = Path(BASE_DIR)
(BASE_DIR / "data").mkdir(parents=True, exist_ok=True)
(BASE_DIR / "artifacts" / "models").mkdir(parents=True, exist_ok=True)
print("Working directory:", BASE_DIR)

In [ ]:
# 3) Upload kaggle.json and download FER2013
import os
import shutil
import zipfile
from google.colab import files

KAGGLE_DIR = Path.home() / ".kaggle"
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)

print("Upload your kaggle.json file now...")
uploaded = files.upload()

if "kaggle.json" not in uploaded:
    raise RuntimeError("Please upload a file named kaggle.json")

kaggle_path = KAGGLE_DIR / "kaggle.json"
with open(kaggle_path, "wb") as f:
    f.write(uploaded["kaggle.json"])
os.chmod(kaggle_path, 0o600)

!pip -q install kaggle

FER_ROOT = BASE_DIR / "data" / "fer2013"
if not (FER_ROOT / "train").exists():
    print("Downloading FER2013 from Kaggle...")
    !kaggle datasets download -d msambare/fer2013 -p {BASE_DIR / "data"} --unzip
    # Kaggle zip usually extracts to fer2013/ with train/ and test/
    extracted = BASE_DIR / "data"
    candidates = list(extracted.glob("**/train"))
    if candidates:
        # If nested, normalize to data/fer2013/train
        parent = candidates[0].parent
        if parent.name != "fer2013":
            target = FER_ROOT
            target.mkdir(parents=True, exist_ok=True)
            if (parent / "train").exists():
                shutil.copytree(parent / "train", FER_ROOT / "train", dirs_exist_ok=True)
            if (parent / "test").exists():
                shutil.copytree(parent / "test", FER_ROOT / "test", dirs_exist_ok=True)
    else:
        # fallback: unzip fer2013.zip manually
        zip_path = BASE_DIR / "data" / "fer2013.zip"
        if zip_path.exists():
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(BASE_DIR / "data")
else:
    print("FER2013 already present at", FER_ROOT)

assert (FER_ROOT / "train").exists(), f"train/ not found under {FER_ROOT}"
assert (FER_ROOT / "test").exists(), f"test/ not found under {FER_ROOT}"
print("FER2013 ready:", FER_ROOT)

In [ ]:
# 4) Config (matches smartshop/mood_model/config.yaml)
import random
import numpy as np

EMOTION_ORDER = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
NUM_CLASSES = 7
IMAGE_SIZE = 224
SEED = 42
BATCH_SIZE = 32  # Colab GPU can handle larger batches
EPOCHS_HEAD = 10
EPOCHS_FINETUNE = 10  # set 0 to skip fine-tuning stage
HEAD_LR = 1e-3
FINETUNE_LR = 1e-5
VAL_SPLIT = 0.15

FER_ROOT = BASE_DIR / "data" / "fer2013"
MODELS_DIR = BASE_DIR / "artifacts" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# 5) FER2013 dataloaders + label verification
from collections import Counter
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_dir = FER_ROOT / "train"
test_dir = FER_ROOT / "test"

train_full = datasets.ImageFolder(str(train_dir), transform=train_transform)
train_for_val = datasets.ImageFolder(str(train_dir), transform=eval_transform)
test_ds = datasets.ImageFolder(str(test_dir), transform=eval_transform)

# Verify labels
expected = set(EMOTION_ORDER)
found = set(train_full.class_to_idx.keys())
if expected != found:
    raise ValueError(f"Label mismatch. Expected {EMOTION_ORDER}, found {sorted(found)}")

val_size = int(len(train_full) * VAL_SPLIT)
train_size = len(train_full) - val_size
g = torch.Generator().manual_seed(SEED)
train_idx, val_idx = random_split(range(len(train_full)), [train_size, val_size], generator=g)

train_ds = Subset(train_full, list(train_idx.indices))
val_ds = Subset(train_for_val, list(val_idx.indices))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("Train samples:", len(train_ds), "| Val:", len(val_ds), "| Test:", len(test_ds))
print("Class mapping:", train_full.class_to_idx)

# Distribution on train split
idx_to_class = {v: k for k, v in train_full.class_to_idx.items()}
counts = Counter()
for i in train_idx.indices:
    _, c = train_full.samples[i]
    counts[idx_to_class[c]] += 1
print("Train class distribution:", dict(counts))

In [ ]:
# 6) MobileNetV3-Small model
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

def create_model(num_classes: int = NUM_CLASSES):
    weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
    model = mobilenet_v3_small(weights=weights)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = torch.nn.Linear(in_features, num_classes)
    return model

def freeze_backbone(model):
    for name, param in model.named_parameters():
        if not name.startswith("classifier"):
            param.requires_grad = False

def unfreeze_last_block(model):
    for param in model.features[-1].parameters():
        param.requires_grad = True
    for param in model.classifier.parameters():
        param.requires_grad = True

model = create_model().to(device)
print(model.__class__.__name__, "ready")

In [ ]:
# 7) Training utilities
from sklearn.metrics import f1_score, accuracy_score, classification_report
import copy

def compute_class_weights(dataset, num_classes: int):
    labels = []
    if isinstance(dataset, Subset):
        base = dataset.dataset
        for i in dataset.indices:
            _, y = base.samples[i]
            labels.append(y)
    else:
        for _, y in dataset.samples:
            labels.append(y)
    counts = np.bincount(labels, minlength=num_classes).astype(np.float32)
    counts[counts == 0] = 1.0
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32)

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    n = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if is_train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)
            if is_train:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * images.size(0)
        n += images.size(0)
        preds = logits.argmax(dim=1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    return {
        "loss": total_loss / max(n, 1),
        "acc": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
    }

class_weights = compute_class_weights(train_ds, NUM_CLASSES).to(device)
print("Class weights:", class_weights.cpu().numpy())

In [ ]:
# 8) Stage 1 — train classification head only
freeze_backbone(model)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=HEAD_LR)

best_val_f1 = -1.0
best_state = None
history = []

for epoch in range(1, EPOCHS_HEAD + 1):
    tr = run_epoch(model, train_loader, criterion, optimizer)
    va = run_epoch(model, val_loader, criterion)
    history.append({"epoch": epoch, "stage": "head", **{f"train_{k}": tr[k] for k in tr}, **{f"val_{k}": va[k] for k in va}})
    print(f"[Head {epoch}/{EPOCHS_HEAD}] train loss={tr['loss']:.4f} acc={tr['acc']:.4f} | val loss={va['loss']:.4f} acc={va['acc']:.4f} macro_f1={va['macro_f1']:.4f}")
    if va["macro_f1"] > best_val_f1:
        best_val_f1 = va["macro_f1"]
        best_state = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)
print("Best head-only val macro F1:", best_val_f1)

In [ ]:
# 9) Stage 2 — optional fine-tune last block
if EPOCHS_FINETUNE > 0:
    unfreeze_last_block(model)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=FINETUNE_LR)

    for epoch in range(1, EPOCHS_FINETUNE + 1):
        tr = run_epoch(model, train_loader, criterion, optimizer)
        va = run_epoch(model, val_loader, criterion)
        history.append({"epoch": epoch, "stage": "finetune", **{f"train_{k}": tr[k] for k in tr}, **{f"val_{k}": va[k] for k in va}})
        print(f"[FT {epoch}/{EPOCHS_FINETUNE}] train loss={tr['loss']:.4f} acc={tr['acc']:.4f} | val loss={va['loss']:.4f} acc={va['acc']:.4f} macro_f1={va['macro_f1']:.4f}")
        if va["macro_f1"] > best_val_f1:
            best_val_f1 = va["macro_f1"]
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    print("Best after fine-tune val macro F1:", best_val_f1)
else:
    print("Skipping fine-tune stage (EPOCHS_FINETUNE=0)")

In [ ]:
# 10) Test evaluation
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

test_acc = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average="macro")
print(f"TEST accuracy={test_acc:.4f} macro_f1={test_f1:.4f}")
print(classification_report(all_labels, all_preds, target_names=EMOTION_ORDER))

In [ ]:
# 11) Save best_model.pt (for local smartshop/mood_model)
import json

checkpoint = {
    "model_state_dict": best_state,
    "emotion_order": EMOTION_ORDER,
    "class_to_idx": train_full.class_to_idx,
    "image_size": IMAGE_SIZE,
    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std": IMAGENET_STD,
    "backbone": "mobilenet_v3_small",
    "num_classes": NUM_CLASSES,
    "best_val_macro_f1": float(best_val_f1),
    "test_accuracy": float(test_acc),
    "test_macro_f1": float(test_f1),
}

best_path = MODELS_DIR / "best_model.pt"
torch.save(checkpoint, best_path)
print("Saved:", best_path)

metrics_path = BASE_DIR / "artifacts" / "metrics.json"
with open(metrics_path, "w") as f:
    json.dump({
        "best_val_macro_f1": best_val_f1,
        "test_accuracy": test_acc,
        "test_macro_f1": test_f1,
        "history": history,
    }, f, indent=2)
print("Metrics:", metrics_path)

In [ ]:
# 12) Download best_model.pt to your laptop
from google.colab import files

print("Downloading best_model.pt ...")
files.download(str(best_path))
print("\nPlace the file here on your Mac:")
print("smartshop/mood_model/artifacts/models/best_model.pt")

## Optional — custom participant dataset (fine-tune later)

If you have your own dataset zip with structure:

```
dataset/
  participant_1/angry/*.jpg
  participant_2/happy/*.jpg
```

1. Set `USE_CUSTOM = True`
2. Upload zip when prompted
3. Run the cell below (participant-level split can be added in local `mood_model` pipeline next)

In [ ]:
# Optional custom dataset upload (skip if not ready)
USE_CUSTOM = False

if USE_CUSTOM:
    print("Upload your custom dataset zip (e.g. dataset.zip)...")
    up = files.upload()
    zip_name = next(iter(up.keys()))
    custom_root = BASE_DIR / "data" / "custom"
    custom_root.mkdir(parents=True, exist_ok=True)
    zip_path = BASE_DIR / "data" / zip_name
    with open(zip_path, "wb") as f:
        f.write(up[zip_name])
    import zipfile
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(custom_root)
    print("Extracted to:", custom_root)
    print("Next: implement participant-level fine-tune in local mood_model or extend this notebook.")
else:
    print("Custom dataset step skipped.")